In [49]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import sklearn.neural_network
import matplotlib.pyplot as plt

# Lab 9 - Multi-layer Perceptron Forward Pass & Backpropagation

## Part I
For this exercise you will implement a simple 2-layer perceptron with the forward pass and the backpropagation to learn the weights

For the first part you'll build and train a 2-layer neural network that predicts the prices of houses, using the usual Boston housing dataset.

In [35]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
boston = pd.read_table("housing.txt", names=housing_names, sep="\s+")

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
D:\Users\C3007807\AppData\Local\Temp\ipykernel_20036\3446792430.py:2: SyntaxWarning: invalid escape sequence '\s'
  boston = pd.read_table("housing.txt", names=housing_names, sep="\s+")


As usual, consider the MEDV as your target variable. 
* Split the data into training, validation and testing (70,15,15)%
* Experiment with different number of neurons per layer for your network, using the validation set

In [36]:
X = boston.values[:,:-1]
y = boston.values[:,-1]

In [37]:
X_train, X_aux, y_train, y_aux = train_test_split(X, y, train_size= 0.7)
X_test, X_val, y_test, y_val = train_test_split(X_aux, y_aux, train_size= 0.5)


In [38]:
X_train.shape

(354, 13)

In [39]:
y_train.shape

(354,)

In [40]:
class ActivationFunction:
    
    def __init__(self, function, derivative = None):
        self.function = function
        self.derivative = derivative
        
def sigmoid(a): return 1/(1 + np.exp(-a))
def sigmoid_prime(a): return sigmoid(a) * (1 - sigmoid(a))

sigmoid_activation = ActivationFunction(sigmoid, sigmoid_prime)

def softmax(a):
    exp_a = np.exp(a)
    return exp_a / exp_a.sum(axis=1, keepdims=True)

def identity(a): return a

In [41]:
def sse(target, y_prev): return np.sum((target - y_prev)**2) * 0.5

def cross_entropy_multiclass(target, y_prev): 
    n = len(target)
    return - np.sum(np.multiply(target, np.log(y_prev))) / n

In [42]:
class MultilayerPerceptron:

    def __init__(self, hidden_activation: ActivationFunction, output_activation, sizes: list):
        """A lista "sizes" deve conter todos as dimensões das camadas da nossa rede, desde a dimensão de entrada
        até a de saída. Tanto os pesos, quanto os biases estarão na matriz de parâmetros."""

        self.num_layers = len(sizes) - 1
        self.sizes = sizes
        self.hidden_activation = hidden_activation.function
        self.hidden_activation_prime = hidden_activation.derivative
        self.output_activation = output_activation
        self.parameters = [np.random.randn(x + 1,y) for x,y in zip(sizes[:-1], sizes[1:])]
    

    def _initialize_parameters(self):
        self.parameters = [np.random.randn(x + 1,y) for x,y in zip(self.sizes[:-1], self.sizes[1:])]
    

    @staticmethod
    def _add_ones(X):
        """Essa função vai nos permitir adicionar uma coluna de uns aos nossos dados, para conseguirmos 
        multiplicar pelos biases."""
    
        return np.column_stack((np.ones(X.shape[0]), X))
    

    def _forward_pass(self, X):
        z = self._add_ones(X)
        Zs = [z]
        As = []
        
        #Hidden Layers
        for w in self.parameters[:-1]:
            a = z @ w
            As.append(a)
            
            z = self.hidden_activation(a) 
            z = self._add_ones(z)
            Zs.append(z)

        #Output Layer
        a = z@self.parameters[-1]
        As.append(a)
        y = self.output_activation(a)

        return y, Zs, As
    

    def _fast_forward_pass(self, X):
        """Realiza o forward_pass sem ficar armazendo os valores necessários para o backpropagation."""
        z = self._add_ones(X)
        
        #Hidden Layers
        for w in self.parameters[:-1]:
            a = z @ w
            
            z = self.hidden_activation(a) 
            z = self._add_ones(z)

        #Output Layer
        a = z@self.parameters[-1]
        y = self.output_activation(a)

        return y
        

    @staticmethod
    def _output_derivative(y_prev, target):
        return y_prev - target
    
    
    def compute_loss(self, y_prev, target): pass
    

    def fit(self, 
            X_train, 
            y_train, 
            X_val,
            y_val,
            batch_size, 
            lr, 
            epochs=1, 
            weight_decay = 0.0):
        
        """Treina os parâmetros da nossa rede utilizando o método de mini-batch. Os casos do Stochastic Gradient Descent (SGD)
        e do método batch, são os casos especiais em que batch_size = 1 e batch_size = X.shape[0] (número de data points), respectivamente."""

        n_samples = X_train.shape[0]
        loss_history_train = []
        loss_history_val = [] 

        for epoch in range(epochs):

            for start in range(0, n_samples, batch_size):
                X_batch = X_train[start : start + batch_size]
                y_batch = y_train[start : start + batch_size]

                delta_w = self.backpropagation(X_batch, y_batch)

                for i, dw in enumerate(delta_w):
                    
                    if weight_decay > 0:
                        regularization = np.zeros_like(self.parameters[i])
                        regularization[1:] = weight_decay * self.parameters[i][1:] 
                        dw = dw + regularization
                    
                    self.parameters[i] -= lr * dw

            y_pred_train = self._fast_forward_pass(X_train)
            y_pred_val = self._fast_forward_pass(X_val)
            loss_history_train.append(self.compute_loss(y_pred_train, y_train))
            loss_history_val.append(self.compute_loss(y_pred_val, y_val))

        return loss_history_train, loss_history_val

                    

In [43]:
class MultilayerPerceptronRegression(MultilayerPerceptron):
    
    def __init__(self, hidden_activation, output_activation, sizes):
        super().__init__(hidden_activation, output_activation, sizes)
    
    def backpropagation(self, X, y):
        n = X.shape[0]
        delta_w = [np.zeros_like(w) for w in self.parameters]

        y_pred, Zs, As = self._forward_pass(X)

        n_out = y_pred.shape[1]
        if n_out == 1:
            y_enc = y.reshape(-1, 1)          

        delta = self._output_derivative(y_pred, y_enc)  
        delta_w[-1] = Zs[-1].T @ delta / n

        for l in range(2, self.num_layers):
            delta = (delta @ self.parameters[-l + 1][1:].T) * self.hidden_activation_prime(As[-l])
            
            delta_w[-l] = Zs[-l].T @ delta / n

        return delta_w
    
    def predict(self, X):
        y_pred = self._fast_forward_pass(X)
        
        if y_pred.shape[1] == 1:
            y_pred = y_pred.flatten()
            
        return y_pred
    
    @staticmethod
    def sse(target, y_prev): return np.sum((target - y_prev)**2) * 0.5

    @staticmethod
    def rmse(target, y_prev): return np.sqrt(np.mean((target - y_prev)**2))

    def compute_loss(self, y_prev, target):
        return self.rmse(y_prev, target)

In [44]:
class MultilayerPerceptronClassification(MultilayerPerceptron):

    def __init__(self, hidden_activation, output_activation, sizes):
        super().__init__(hidden_activation, output_activation, sizes)
        self.num_classes = sizes[-1]
    
    @staticmethod
    def _one_hot(y, num_classes):
        """Converte labels inteiros em matriz com uns nas classes correspondentes"""
        
        Y = np.zeros((len(y), num_classes))
        Y[np.arange(len(y)), y] = 1
        return Y
        

    @staticmethod
    def _output_derivative(y_prev, target):
        return y_prev - target
    
    def backpropagation(self, X, y):
        n      = X.shape[0]
        delta_w = [np.zeros_like(w) for w in self.parameters]

        y_pred, Zs, As = self._forward_pass(X)

        if y.ndim == 1:
            y_target  = self._one_hot(y, self.num_classes)         

        delta = self._output_derivative(y_pred, y_target)  
        delta_w[-1] = Zs[-1].T @ delta / n

        for l in range(2, self.num_layers):
            delta = (delta @ self.parameters[-l + 1][1:].T) * self.hidden_activation_prime(As[-l])
            
            delta_w[-l] = Zs[-l].T @ delta / n

        return delta_w
    
    def predict(self, X):
        return self._fast_forward_pass(X)
    
    def predict_classes(self, X):
        """Retorna o índice da classe com maior probabilidade."""
        
        y_pred = self._fast_forward_pass(X)
        return np.argmax(y_pred, axis=1)
    
    def cross_entropy_multiclass(self, target, y_prev):
        t = self._one_hot(target, self.num_classes)
        n = len(target)
        return - np.sum(np.multiply(t, np.log(y_prev))) 
    
    def compute_loss(self, y_prev, target):
        return self.cross_entropy_multiclass(y_prev, target)

In [72]:
MLP = MultilayerPerceptronRegression(sigmoid_activation, identity, [13, 30, 1])

In [73]:
lht, lhv = MLP.fit(X_train, y_train, X_val, y_val, 1, 0.1, 50, 1e-3)

D:\Users\C3007807\AppData\Local\Temp\ipykernel_20036\1096152351.py:7: RuntimeWarning: overflow encountered in exp
  def sigmoid(a): return 1/(1 + np.exp(-a))


In [83]:
y_prev = MLP.predict(X_train)

In [84]:
MLP.rmse(y_train, y_prev)

np.float64(14.708198866960846)

## Part II 

For this exercise you will build and train a 2-layer neural network that predicts the exact digit from a hand-written image, using the MNIST dataset. 
For this exercise, add weight decay to your network.

In [109]:
from sklearn.datasets import load_digits

In [110]:
digits = load_digits()

In [312]:
X = digits.data
y = digits.target

In [313]:
X.shape

(1797, 64)

Again, you will split the data into training, validation and testing.

In [314]:
X_train, X_aux, y_train, y_aux = train_test_split(X, y, train_size= 0.7)
X_test, X_val, y_test, y_val = train_test_split(X, y, train_size= 0.5)

In [549]:
MLP = MultilayerPerceptronClassification(sigmoid_activation, softmax, [64, 14, 10])

In [550]:
MLP.fit(X_train, y_train, 10, 0.1, 50, 1e-3)

In [551]:
y_pred = MLP.predict(X_val)

In [552]:
MLP.cross_entropy_multiclass(y_val, y_pred)

np.float64(953.2873212673422)

In [547]:
y_val

array([8, 6, 6, 9, 3, 0, 9, 5, 0, 7, 1, 8, 5, 7, 2, 9, 7, 7, 3, 4, 9, 6,
       5, 5, 8, 6, 6, 6, 8, 9, 2, 3, 6, 6, 7, 5, 6, 7, 8, 4, 3, 9, 2, 0,
       2, 2, 1, 7, 7, 0, 9, 9, 1, 0, 0, 1, 7, 0, 1, 0, 2, 3, 4, 9, 9, 4,
       7, 4, 1, 7, 9, 0, 6, 6, 6, 7, 3, 7, 6, 8, 6, 0, 6, 7, 2, 1, 1, 4,
       2, 7, 8, 3, 3, 2, 8, 6, 3, 5, 2, 6, 0, 3, 6, 5, 4, 9, 8, 6, 6, 8,
       5, 1, 2, 1, 6, 1, 5, 7, 5, 3, 2, 1, 1, 7, 3, 6, 5, 3, 4, 9, 3, 7,
       1, 3, 3, 6, 0, 6, 9, 1, 8, 5, 8, 5, 3, 4, 6, 3, 2, 0, 9, 9, 1, 5,
       0, 1, 1, 5, 2, 8, 9, 2, 4, 2, 5, 4, 6, 0, 0, 6, 7, 9, 4, 7, 3, 8,
       0, 7, 0, 1, 0, 7, 2, 6, 6, 6, 5, 8, 3, 1, 2, 7, 6, 7, 2, 9, 6, 7,
       7, 4, 7, 6, 9, 0, 3, 1, 1, 4, 7, 5, 8, 0, 8, 4, 9, 7, 3, 0, 4, 2,
       7, 7, 9, 5, 7, 6, 7, 0, 9, 2, 9, 3, 7, 3, 2, 7, 1, 5, 7, 9, 1, 6,
       0, 1, 6, 4, 7, 7, 0, 0, 3, 7, 9, 6, 8, 8, 1, 3, 3, 0, 7, 4, 4, 4,
       3, 9, 1, 7, 5, 3, 1, 7, 6, 8, 0, 2, 6, 7, 4, 6, 1, 4, 8, 9, 1, 6,
       2, 9, 8, 0, 0, 7, 1, 3, 6, 7, 1, 4, 6, 9, 9,

In [548]:
MLP.predict_classes(X_val)

array([5, 6, 6, 9, 3, 0, 9, 5, 6, 8, 1, 5, 5, 8, 1, 9, 7, 7, 2, 4, 5, 6,
       5, 5, 1, 6, 6, 6, 8, 3, 2, 5, 6, 6, 7, 5, 6, 7, 8, 4, 3, 9, 2, 0,
       2, 2, 1, 8, 7, 0, 9, 9, 1, 0, 0, 1, 7, 0, 8, 0, 2, 8, 4, 9, 9, 4,
       7, 4, 1, 8, 9, 0, 6, 6, 6, 7, 9, 7, 6, 1, 6, 0, 6, 8, 2, 1, 8, 4,
       2, 7, 8, 3, 3, 3, 8, 6, 3, 5, 8, 6, 0, 5, 6, 5, 4, 9, 8, 6, 8, 8,
       5, 1, 2, 1, 6, 1, 5, 1, 5, 3, 2, 1, 1, 8, 5, 6, 5, 3, 4, 9, 3, 7,
       1, 3, 5, 6, 0, 6, 9, 8, 8, 5, 8, 5, 3, 4, 6, 9, 2, 0, 8, 9, 1, 5,
       0, 1, 1, 5, 2, 8, 9, 2, 4, 2, 5, 4, 6, 0, 0, 6, 7, 9, 4, 7, 9, 8,
       0, 7, 0, 1, 0, 7, 2, 6, 6, 6, 5, 8, 3, 1, 2, 7, 6, 7, 8, 9, 6, 7,
       7, 4, 7, 6, 9, 0, 5, 9, 1, 7, 7, 5, 8, 0, 8, 4, 9, 7, 2, 0, 4, 1,
       8, 7, 9, 5, 4, 6, 8, 1, 9, 2, 9, 3, 7, 3, 2, 3, 1, 2, 7, 9, 1, 6,
       0, 8, 6, 4, 7, 7, 0, 0, 9, 4, 9, 6, 8, 8, 8, 8, 3, 0, 7, 4, 4, 4,
       3, 6, 1, 7, 5, 3, 1, 7, 6, 8, 0, 2, 6, 7, 4, 6, 1, 4, 8, 9, 1, 6,
       8, 9, 8, 0, 0, 7, 1, 3, 6, 7, 1, 4, 6, 9, 9,